In [ ]:
import warnings

warnings.filterwarnings("ignore")
import os

import matplotlib.pyplot as plt
import pandas as pd

from guild.tools.ligand_properties import (
    assign_properties,
    compound_filter,
    generate_equal_mw_distributions,
)
from guild.transformers.chembl import get_assays_locally, get_decoys_locally

# Get protein info

In [ ]:
support_dir = "../../guild/support"

os.makedirs(f"{support_dir}/binders", exist_ok=True)
os.makedirs(f"{support_dir}/decoys", exist_ok=True)

In [ ]:
gpcrdb_mapping = pd.read_csv(f"{support_dir}/uniprot_gpcrdb_mapping.txt", sep=" ")
gpcrdb_mapping.head(2)

In [ ]:
uniprot_gene_list = set(gpcrdb_mapping["uniprot_id"].tolist())
len(uniprot_gene_list)

# Update binders to ChEMBL 36

In [ ]:
assay_df = get_assays_locally(uniprot_gene_list, min_mol_wt=250, max_mol_wt=450)

In [ ]:
# Filter out compounds with pchembl value <= 0
assay_df = assay_df[assay_df["pchembl_value"] > 0]

# Assign activity based on pchembl value 6
assay_df["activity"] = assay_df["pchembl_value"].apply(
    lambda x: "weak-binder" if x < 6 else "strong-binder"
)
chembl_version = assay_df["chembl_version"].values[0]
chembl_version

In [ ]:
BINDER_FILE = f"{support_dir}/binders/chembl_binders_{chembl_version}.tsv"

if not os.path.exists(BINDER_FILE):
    assay_df_filtered = assay_df[
        [
            "protein_symbol",
            "activity",
            "chembl_id",
            "smiles",
            "pchembl_value",
        ]
    ]
    assay_df_filtered["chembl_id"] = "CHEMBL" + assay_df_filtered["chembl_id"].astype(
        str
    )
    assay_df_filtered.rename(columns={"protein_symbol": "uniprot_id"}, inplace=True)

    assay_df_filtered = pd.merge(
        assay_df_filtered, gpcrdb_mapping, how="outer", on="uniprot_id"
    )

    assay_df_filtered.to_csv(
        f"{support_dir}/binders/chembl_binders_{chembl_version}.tsv",
        sep="\t",
        index=False,
    )
else:
    assay_df_filtered = pd.read_csv(BINDER_FILE, sep="\t")

# Update decoys to ChEMBL 36

In [ ]:
MIN_MW = 250
MAX_MW = 450
MAX_RING_SIZE = 10

In [ ]:
decoy_df = get_decoys_locally(
    min_mol_wt=MIN_MW, max_mol_wt=MAX_MW, version="chembl_36"
)  # without the version flag, the latest chembl data is pulled
decoy_df.head()

### Add molecular properties to the df

In [ ]:
if not os.path.exists(f"{support_dir}/decoys/{chembl_version}_with_properties.tsv"):
    decoy_df_with_properties = assign_properties(
        decoy_df[["chembl_id", "canonical_smiles"]]
    )
    decoy_df_with_properties["scaffold_smiles"] = decoy_df_with_properties[
        "scaffold_smiles"
    ].replace("", None)

    decoy_df_with_properties.to_csv(
        f"{support_dir}/decoys/{chembl_version}_with_properties.tsv",
        sep="\t",
        index=False,
    )
else:
    decoy_df_with_properties = pd.read_csv(
        f"{support_dir}/decoys/{chembl_version}_with_properties.tsv",
        sep="\t",
        low_memory=False,
    )

decoy_df_with_properties.head()

## Subset to match the same properties as other ligands

In [ ]:
all_decoy_df = pd.merge(
    decoy_df,
    decoy_df_with_properties,
    left_on="chembl_id",
    right_on="id",
    how="inner",
)
all_decoy_df.drop(columns=["id"], inplace=True)

In [ ]:
all_decoy_df.to_csv(
    f"{support_dir}/decoys/{chembl_version}_decoys_with_properties.tsv",
    sep="\t",
    index=False,
)

In [ ]:
all_decoy_df["molecular_weight"].plot.hist(bins=100)
plt.show()

### Subset the df based on scaffolds and MW in the distribution

In [ ]:
if os.path.exists(f"{support_dir}/decoys/{chembl_version}_decoys_filtered.tsv"):
    decoys_filtered = pd.read_csv(
        f"{support_dir}/decoys/{chembl_version}_decoys_filtered.tsv", sep="\t"
    )
else:
    decoys_filtered = compound_filter(
        all_decoy_df,
        min_MW=MIN_MW,
        max_MW=MAX_MW,
        scaffold_col="scaffold_smiles",
        ro5_fulfilled_col="ro5_fulfilled",
        largest_ring_size_col="max_ring_size",
        max_per_scaffold=5,
        seed=42,
        MW_column="molecular_weight",
        max_ring_size=MAX_RING_SIZE,
    )

    decoys_filtered.to_csv(
        f"{support_dir}/decoys/{chembl_version}_decoys_filtered.tsv",
        sep="\t",
        index=False,
    )

# Generate decoys sublists

In [ ]:
# Decoys with 1000 samples
decoys_sample_1000 = generate_equal_mw_distributions(
    df_1=decoys_filtered,
    mw_column="molecular_weight",
    lipinski_column="ro5_fulfilled",
    n_bins=100,
    target_size=1000,
)

decoys_sample_1000.to_csv(
    f"{support_dir}/decoys/{chembl_version}_decoys_1000.tsv",
    sep="\t",
    index=False,
)

In [ ]:
decoys_sample_1000["molecular_weight"].plot.hist(bins=100)
plt.show()

### Generate 100 decoys list

In [ ]:
decoys_sample_100 = generate_equal_mw_distributions(
    df_1=decoys_filtered,
    mw_column="molecular_weight",
    lipinski_column="ro5_fulfilled",
    n_bins=20,
    target_size=100,
)

decoys_sample_100.to_csv(
    f"{support_dir}/decoys/{chembl_version}_decoys_100.tsv",
    sep="\t",
    index=False,
)

In [ ]:
decoys_sample_100["molecular_weight"].plot.hist(bins=20)
plt.show()